# NB7 — Multimodal & Vectors: when one row weighs 10 MB

**Stack:** `deltalake` + DuckDB + NumPy. No model download, no vector DB, no API key.
Maps to slide §11 (AI 2026: Multimodal, Vector & Agent) + deliverable bullet 7.

Parquet and Iceberg were designed for **KB-sized rows, read sequentially, in
batches**. Multimodal AI breaks all three assumptions at once:

1. Rows become **MB** (images, clips, audio)
2. Access becomes **random** — thousands of clips/vectors per second
3. The GPU must be fed continuously; a starved GPU is money on fire

This notebook measures what actually breaks, then builds semantic search
directly on a lakehouse table — and ends with the bug that matters most.

In [1]:
import _setup  # noqa: F401

import time
from pathlib import Path

import duckdb
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import DeltaTable, write_deltalake

from lakehouse import ROOT, du, human, path, reset

import generate_ai_data as gen

DOCS = path("bronze", "docs_multimodal")
if not Path(DOCS).exists():
    gen.main()

docs = DeltaTable(DOCS).to_pyarrow_table()
print(f"Corpus: {docs.num_rows:,} docs, embedding dim={len(docs.column('emb')[0])}")
print(f"Columns: {docs.column_names}")

Corpus: 2,000 docs, embedding dim=256
Columns: ['doc_id', 'title', 'topic', 'subject_id', 'source', 'license', 'consent_train', 'generator', 'blob_uri', 'emb']


### 📝 Nhận xét — Corpus

- 2.000 tài liệu tổng hợp, embedding **256 chiều**, cùng các cột quản trị (`subject_id`, `source`, `license`, `consent_train`, `generator`) và `blob_uri` trỏ tới 200 media frame trong `_lakehouse/blobs/`.
- Điểm mấu chốt của NB7: **vector và các cột quản trị nằm trong cùng một dòng** của bảng lakehouse — mọi phần sau đều dựa trên điều này.


## 1. Inline blob vs pointer — measure, don't assume

The usual advice is "never put blobs in your table." Let's test it properly
instead of repeating it. Two layouts, same 200 media frames:

* **inline**  — a `blob` column of raw bytes
* **pointer** — a `blob_uri` string; bytes live as separate objects

In [2]:
BLOB_DIR = Path(ROOT) / "blobs"
blob_files = sorted(BLOB_DIR.glob("*.bin"))
blobs = [f.read_bytes() for f in blob_files]
N = len(blobs)

inline_tbl = pa.table({
    "doc_id": pa.array(range(N), pa.int64()),
    "topic":  [gen.TOPICS[i % len(gen.TOPICS)] for i in range(N)],
    "blob":   pa.array(blobs, pa.binary()),
})
pointer_tbl = pa.table({
    "doc_id":   pa.array(range(N), pa.int64()),
    "topic":    [gen.TOPICS[i % len(gen.TOPICS)] for i in range(N)],
    "blob_uri": [f"blobs/{f.name}" for f in blob_files],
})

INLINE, POINTER = path("scratch", "media_inline"), path("scratch", "media_pointer")
reset(INLINE, POINTER)
write_deltalake(INLINE, inline_tbl, mode="overwrite")
write_deltalake(POINTER, pointer_tbl, mode="overwrite")

print(f"inline table : {human(du(INLINE)):>10}")
print(f"pointer table: {human(du(POINTER)):>10}   (+ {human(du(BLOB_DIR))} of objects alongside)")
print(f"\nTotal bytes stored is nearly identical — the bytes have to live somewhere.")
print("The difference is not SIZE. It is what a reader is forced to touch.")

inline table :    12.5 MB


pointer table:     4.7 KB   (+ 12.5 MB of objects alongside)

Total bytes stored is nearly identical — the bytes have to live somewhere.
The difference is not SIZE. It is what a reader is forced to touch.


### 📝 Nhận xét — Inline vs pointer: cùng dung lượng

- Inline: bảng **12,5 MB** (200 blob × 64 KB nằm trong cột `blob`). Pointer: bảng chỉ **4,7 KB** nhưng có **12,5 MB object** nằm bên cạnh.
- Tổng byte gần như bằng nhau — dữ liệu phải nằm ở đâu đó. Khác biệt **không phải dung lượng** mà là **reader buộc phải đọc những byte nào** — hai finding bên dưới đo đúng điều đó.


### Finding 1: column pruning already protects analytical scans

A query like `SELECT topic, count(*) GROUP BY topic` never mentions `blob`.
Parquet is columnar, so it reads only the column chunks it needs.

We can prove it exactly from Parquet footer metadata — no timing required.

In [3]:
def column_bytes(table_path: str) -> dict[str, int]:
    """Compressed bytes per column, straight from the Parquet footer."""
    data_file = next(f for f in Path(table_path).rglob("*.parquet") if "_delta_log" not in f.parts)
    md = pq.ParquetFile(data_file).metadata
    out: dict[str, int] = {}
    for rg in range(md.num_row_groups):
        for c in range(md.row_group(rg).num_columns):
            col = md.row_group(rg).column(c)
            out[col.path_in_schema] = out.get(col.path_in_schema, 0) + col.total_compressed_size
    return out


inline_cols = column_bytes(INLINE)
pointer_cols = column_bytes(POINTER)
projected = ["doc_id", "topic"]

inline_scan = sum(inline_cols[c] for c in projected)
pointer_scan = sum(pointer_cols[c] for c in projected)
print("Bytes read by `SELECT topic, count(*) GROUP BY topic`:")
print(f"  inline layout : {human(inline_scan)}  of {human(sum(inline_cols.values()))} total")
print(f"  pointer layout: {human(pointer_scan)}  of {human(sum(pointer_cols.values()))} total")
print(f"\n→ The blob column costs the analytical scan essentially NOTHING.")
print("  Projection pushdown is real. The scary advice is wrong for this case.")

Bytes read by `SELECT topic, count(*) GROUP BY topic`:
  inline layout : 1.2 KB  of 12.5 MB total
  pointer layout: 1.2 KB  of 2.4 KB total

→ The blob column costs the analytical scan essentially NOTHING.
  Projection pushdown is real. The scary advice is wrong for this case.


### 📝 Nhận xét — Finding 1: projection pushdown bảo vệ truy vấn phân tích

- Đọc thẳng **footer Parquet** (`total_compressed_size` từng column chunk) — không cần bấm giờ: `SELECT topic, count(*) GROUP BY topic` chỉ cần cột `doc_id`, `topic` ⇒ **1,2 KB / 12,5 MB** ở layout inline, bằng đúng layout pointer.
- Parquet lưu **theo cột**: column chunk của `blob` không được đọc khi truy vấn không nhắc tới nó. Lời khuyên "không bao giờ để blob trong bảng" **sai** với workload phân tích.


### Finding 2: random single-row access is where it actually breaks

Now fetch **one** frame: `WHERE doc_id = 137`.

Parquet's unit of I/O is the **row group**, not the row. To get one blob you
must read (and decompress) the whole row group that contains it. With a
pointer you issue one `GET` for exactly those bytes.

This — not file size — is the number Lance's "3–35× faster random access"
claim is about.

In [4]:
def row_group_bytes(table_path: str) -> tuple[int, int, int]:
    data_file = next(f for f in Path(table_path).rglob("*.parquet") if "_delta_log" not in f.parts)
    md = pq.ParquetFile(data_file).metadata
    rg0 = md.row_group(0)
    return md.num_row_groups, rg0.num_rows, rg0.total_byte_size


n_rg, rg_rows, rg_bytes = row_group_bytes(INLINE)
one_blob = len(blobs[0])

print(f"inline parquet: {n_rg} row group(s), {rg_rows} rows, {human(rg_bytes)} per group")
print(f"\nFetching ONE frame (doc_id=137):")
print(f"  inline  → must read the row group: {human(rg_bytes)}")
print(f"  pointer → one GET of the object:   {human(one_blob)}")
print(f"  → amplification: {rg_bytes / one_blob:.0f}× more bytes than needed")
print("\nAt 1,000 random frame fetches/sec to feed a GPU, that amplification IS")
print("the GPU-starvation problem. Formats like Lance restructure the file so a")
print("random read costs ~one row, not one row group.")

AMPLIFICATION = rg_bytes / one_blob

inline parquet: 1 row group(s), 200 rows, 12.5 MB per group

Fetching ONE frame (doc_id=137):
  inline  → must read the row group: 12.5 MB
  pointer → one GET of the object:   64.0 KB
  → amplification: 200× more bytes than needed

At 1,000 random frame fetches/sec to feed a GPU, that amplification IS
the GPU-starvation problem. Formats like Lance restructure the file so a
random read costs ~one row, not one row group.


### 📝 Nhận xét — Finding 2: truy cập ngẫu nhiên bị khuếch đại theo row group

- Lấy **1 frame** (64 KB): đơn vị I/O của Parquet là **row group**, không phải dòng ⇒ phải đọc (và giải nén) cả row group chứa nó. File inline chỉ có **1 row group gồm cả 200 dòng = 12,5 MB** ⇒ **khuếch đại 200×** (≥ 5× ✅). Vì mọi blob cùng kích thước, hệ số khuếch đại = **số dòng trong row group**: row group càng lớn, càng tệ.
- Pointer: một `GET` lấy đúng 64 KB.
- Hệ quả: để nuôi GPU 1.000 frame/giây, inline phải đọc ~12 GB/s thay vì ~64 MB/s ⇒ **GPU bị đói dữ liệu**. Đây là lý do có các format như **Lance** (tổ chức file để một lần đọc ngẫu nhiên ≈ một dòng).
- **Kết luận §1:** inline blob ổn cho scan phân tích (nhờ column pruning) nhưng tệ cho truy cập ngẫu nhiên (training/serving) ⇒ chọn layout theo **mẫu truy cập**, không theo kích thước.


## 2. Embeddings as a column: the storage arithmetic

The slide's numbers: `VECTOR(768, FLOAT)` = 3,072 B/row; `INT8` = 768 B/row.
Let's verify the 4× on real files rather than trusting the arithmetic.

In [5]:
emb = np.array(docs.column("emb").to_pylist(), dtype="float32")
n, dim = emb.shape
print(f"embeddings: {n:,} × {dim}  ({human(emb.nbytes)} in memory)")
print(f"per row: float32 = {dim * 4:,} B   int8 = {dim * 1:,} B   (4× smaller)")
print(f"at dim=768: float32 = {768 * 4:,} B   int8 = {768:,} B   ← the slide's numbers")

# int8 symmetric quantization: embeddings are unit vectors, so values ∈ [-1, 1]
SCALE = 127.0
emb_i8 = np.clip(np.round(emb * SCALE), -127, 127).astype("int8")

f32_tbl = pa.table({"doc_id": docs.column("doc_id"), "emb": docs.column("emb")})
i8_tbl = pa.table({
    "doc_id": docs.column("doc_id"),
    "emb": pa.FixedSizeListArray.from_arrays(pa.array(emb_i8.ravel(), pa.int8()), dim),
})
F32, I8 = path("scratch", "emb_f32"), path("scratch", "emb_int8")
reset(F32, I8)
write_deltalake(F32, f32_tbl, mode="overwrite")
write_deltalake(I8, i8_tbl, mode="overwrite")

print(f"\nOn disk (Parquet, compressed):")
print(f"  float32: {human(du(F32))}")
print(f"  int8   : {human(du(I8))}   →  {du(F32) / max(du(I8), 1):.1f}× smaller")

embeddings: 2,000 × 256  (2.0 MB in memory)
per row: float32 = 1,024 B   int8 = 256 B   (4× smaller)
at dim=768: float32 = 3,072 B   int8 = 768 B   ← the slide's numbers



On disk (Parquet, compressed):
  float32: 2.6 MB
  int8   : 451.9 KB   →  5.8× smaller


In [6]:
# Vì sao int8 nhỏ hơn > 4×? Đọc footer Parquet của cột emb
for label, p in [("float32", F32), ("int8", I8)]:
    f = next(x for x in Path(p).rglob("*.parquet") if "_delta_log" not in x.parts)
    col = pq.ParquetFile(f).metadata.row_group(0).column(1)
    print(f"{label:8s} codec={col.compression:7s} encodings={col.encodings}  "
          f"uncompressed={human(col.total_uncompressed_size):>9}  compressed={human(col.total_compressed_size):>9}")
print(f"distinct int8 values: {len(np.unique(emb_i8))}  |  distinct float32 values: {len(np.unique(emb)):,}")

float32  codec=SNAPPY  encodings=('PLAIN', 'RLE', 'RLE_DICTIONARY')  uncompressed=   2.6 MB  compressed=   2.5 MB
int8     codec=SNAPPY  encodings=('PLAIN', 'RLE', 'RLE_DICTIONARY')  uncompressed= 448.6 KB  compressed= 439.3 KB
distinct int8 values: 71  |  distinct float32 values: 510,244


### 📝 Nhận xét — Embedding là một cột: int8 nhỏ hơn 5,8×

- Lượng tử hoá đối xứng `round(emb × 127)` (hợp lệ vì embedding là vector đơn vị ⇒ mỗi phần tử ∈ [−1, 1]). Lý thuyết: 4 B → 1 B mỗi phần tử = **4×** (dim 768: 3.072 B → 768 B/dòng như slide).
- Trên đĩa: **2,6 MB → 451,9 KB = 5,8×** (≥ 3× ✅), lớn hơn 4×. Footer Parquet cho thấy lý do **không nằm ở codec snappy** (snappy gần như không giảm được gì: 2,6 → 2,5 MB và 448,6 → 439,3 KB) mà ở **encoding**: kích thước *chưa nén* đã chênh ~5,9×. int8 chỉ có **71 giá trị khác nhau** ⇒ dictionary encoding (`RLE_DICTIONARY`) mã hoá mỗi phần tử bằng chỉ số nhỏ; float32 có **510.244 giá trị khác nhau** trên 512.000 phần tử ⇒ dictionary vô dụng, rơi về `PLAIN` 4 B/phần tử.


## 3. Semantic search *is* a SQL query now

DuckDB ships `array_cosine_similarity` in core — no extension, no download,
no network. The embedding lives in the table, so retrieval is a join away
from every other column you own (topic, license, consent, timestamps).

In [7]:
con = duckdb.connect()
con.register("docs", docs)

# ⚠️ Gotcha worth knowing: we WROTE `fixed_size_list<float>[256]`, but the Delta
# protocol has no fixed-width vector type — only `array<element>`. So the column
# comes back as a variable-length `list<float>` and we must cast it before
# DuckDB's fixed-size array functions will bind:
print("arrow type on read:", docs.schema.field("emb").type, " → cast to FLOAT[dim] at query time")
print("This lab casts the variable-length list to a fixed-size DuckDB array before vector queries.\n")

query_vec = emb[7].tolist()          # pretend this came from an encoder
t0 = time.perf_counter()
hits = con.sql(f"""
    SELECT doc_id, title, topic,
           array_cosine_similarity(emb::FLOAT[{dim}], {query_vec}::FLOAT[{dim}]) AS sim
    FROM docs
    ORDER BY sim DESC
    LIMIT 5
""").fetchall()
sql_ms = (time.perf_counter() - t0) * 1000

print(f"Query doc: {docs.column('title')[7]}  (topic={docs.column('topic')[7]})")
print(f"\n{'doc_id':>7}  {'topic':<12} {'sim':>6}  title")
for doc_id, title, topic, sim in hits:
    print(f"{doc_id:>7}  {topic:<12} {sim:6.3f}  {title}")
print(f"\nbrute-force scan over {n:,} vectors: {sql_ms:.1f} ms")

arrow type on read:

 list<element: float>  → cast to FLOAT[dim] at query time
This lab casts the variable-length list to a fixed-size DuckDB array before vector queries.

Query doc: storage-note-00007  (topic=storage)

 doc_id  topic           sim  title
      7  storage       1.000  storage-note-00007
   1703  storage       0.779  storage-note-01703
   1200  storage       0.777  storage-note-01200
    766  storage       0.776  storage-note-00766
   1250  storage       0.768  storage-note-01250

brute-force scan over 2,000 vectors: 29.0 ms


### 📝 Nhận xét — Semantic search là một câu SQL

- `array_cosine_similarity` có sẵn trong core DuckDB — không extension, không mạng, không vector DB.
- ⚠️ **Bẫy kiểu dữ liệu:** ghi `fixed_size_list<float>[256]` nhưng Delta không có kiểu vector cố định chiều ⇒ đọc lên thành **`list<float>`** độ dài thay đổi ⇒ phải cast `emb::FLOAT[256]` thì hàm mới bind được.
- Truy vấn bằng vector của doc 7 (topic `storage`): top-5 **đều là `storage`** (sim 0,77–0,78 cho các hàng xóm) ✅. Kết quả số 1 là chính doc 7 (sim = 1,000) vì truy vấn lấy từ corpus.
- Brute-force quét 2.000 vector mất ~29 ms.


### Filtered semantic search — the thing a standalone vector DB struggles with

"Find similar docs **that we are actually licensed to train on**" is one
query here, because the vector and the governance columns are in one row.

In [8]:
legal = con.sql(f"""
    SELECT doc_id, topic, license,
           array_cosine_similarity(emb::FLOAT[{dim}], {query_vec}::FLOAT[{dim}]) AS sim
    FROM docs
    WHERE consent_train AND license <> 'unknown'
    ORDER BY sim DESC
    LIMIT 5
""").fetchall()
for doc_id, topic, lic, sim in legal:
    print(f"{doc_id:>7}  {topic:<12} {lic:<12} {sim:6.3f}")
print("\nNo sync job, no ID reconciliation, no 'is this vector still valid?'")

   1703  storage      proprietary   0.779
   1200  storage      user-owned    0.777
    766  storage      commercial    0.776
   1250  storage      user-owned    0.768
    570  storage      commercial    0.768

No sync job, no ID reconciliation, no 'is this vector still valid?'


In [9]:
# Vì sao doc 7 (chính truy vấn, sim=1.000) không có trong kết quả lọc?
print(con.sql("SELECT doc_id, license, consent_train FROM docs WHERE doc_id = 7"))
print(con.sql("""SELECT count(*) FILTER (WHERE consent_train AND license <> 'unknown') AS trainable,
                        count(*) FILTER (WHERE NOT (consent_train AND license <> 'unknown')) AS excluded
                 FROM docs"""))

┌────────┬─────────┬───────────────┐
│ doc_id │ license │ consent_train │
│ int64  │ varchar │    boolean    │
├────────┼─────────┼───────────────┤
│      7 │ unknown │ false         │
└────────┴─────────┴───────────────┘

┌───────────┬──────────┐
│ trainable │ excluded │
│   int64   │  int64   │
├───────────┼──────────┤
│      1666 │      334 │
└───────────┴──────────┘



### 📝 Nhận xét — Filtered semantic search

- Thêm `WHERE consent_train AND license <> 'unknown'` vào cùng câu truy vấn: kết quả vẫn toàn `storage`, kèm license hợp lệ (proprietary, user-owned, commercial).
- **Doc 7 — chính tài liệu dùng làm truy vấn (sim = 1,000) — biến mất** khỏi kết quả lọc: nó có `license = 'unknown'`, `consent_train = false`. Bộ lọc tự loại tài liệu không được phép dùng — cùng với **334/2.000** tài liệu khác.
- Đây là thứ một vector DB đứng riêng làm khó: governance nằm ở hệ thống khác ⇒ phải đồng bộ metadata hoặc lọc sau (post-filter, có thể trả thiếu kết quả). Ở đây vector và governance **cùng một dòng** ⇒ một truy vấn, không sync job, không đối chiếu ID.


### Honest scaling check: brute force is not a serving path

The slide is explicit — in-table brute force suits *analytical* semantic
queries and *offline* recall measurement, not sub-100 ms online serving.

In [10]:
for size in (n, n * 50, n * 500):
    est = sql_ms * size / n
    verdict = "fine" if est < 100 else ("borderline" if est < 1000 else "NOT a serving path")
    print(f"  {size:>10,} vectors → ~{est:8.1f} ms   {verdict}")
print("\nRule from the slide: vector DB = a rebuildable DERIVED INDEX.")
print("                     lakehouse = the SYSTEM-OF-RECORD.")

       2,000 vectors → ~    29.0 ms   fine
     100,000 vectors → ~  1449.1 ms   NOT a serving path
   1,000,000 vectors → ~ 14490.8 ms   NOT a serving path

Rule from the slide: vector DB = a rebuildable DERIVED INDEX.
                     lakehouse = the SYSTEM-OF-RECORD.


### 📝 Nhận xét — Brute force không phải serving path

- Ngoại suy tuyến tính từ ~29 ms / 2.000 vector: 100K ≈ 1,4 s, 1M ≈ 14,5 s ⇒ **không dùng được cho serving online < 100 ms**. Phép ngoại suy thô (lần đo đầu gồm cả chi phí khởi tạo/cast) nhưng kết luận vững vì brute force là O(n).
- Phân vai: **lakehouse = system-of-record** (nơi lưu sự thật, quản trị vòng đời); **vector DB = index phái sinh có thể dựng lại** (phục vụ ANN tốc độ cao). Brute force trong bảng phù hợp cho truy vấn phân tích và đo recall offline.


## 4. What does int8 quantization cost you in recall?

4× less storage is only a win if retrieval quality survives. Measure it:
float32 top-10 is ground truth, int8 top-10 is the candidate.

In [11]:
def topk(matrix: np.ndarray, queries: np.ndarray, k: int = 10) -> np.ndarray:
    sims = queries @ matrix.T
    return np.argsort(-sims, axis=1)[:, :k]


rng = np.random.default_rng(0)
q_idx = rng.choice(n, size=100, replace=False)
emb_i8_deq = emb_i8.astype("float32") / SCALE
emb_i8_deq /= np.linalg.norm(emb_i8_deq, axis=1, keepdims=True)

gold = topk(emb, emb[q_idx])
cand = topk(emb_i8_deq, emb_i8_deq[q_idx])
recall = np.mean([len(set(g) & set(c)) / len(g) for g, c in zip(gold, cand)])

# Exact-ID recall is harsh: it counts a swap between two docs that are equally
# relevant as a miss. For RAG, what usually matters is whether the retrieved
# docs are still ABOUT the right thing. Measure both.
topics_arr = np.array(docs.column("topic").to_pylist())
topic_fidelity = np.mean([
    (topics_arr[c] == topics_arr[q]).mean() for q, c in zip(q_idx, cand)
])

print(f"recall@10 (exact doc IDs), int8 vs float32: {recall:.3f}")
print(f"topic fidelity of int8 top-10:              {topic_fidelity:.3f}")
print(f"storage saved: {(1 - du(I8) / du(F32)) * 100:.0f}%")
print(f"""
→ int8 loses ~{(1 - recall) * 100:.0f}% of exact IDs but {topic_fidelity * 100:.0f}% of results are still
  on-topic. The "misses" are swaps between near-equivalent neighbours, which
  is why exact-ID recall UNDERSTATES quantization quality for RAG.
  Measure both on YOUR corpus before shipping — the trade is corpus-dependent.""")

recall@10 (exact doc IDs), int8 vs float32: 0.904
topic fidelity of int8 top-10:              1.000
storage saved: 83%

→ int8 loses ~10% of exact IDs but 100% of results are still
  on-topic. The "misses" are swaps between near-equivalent neighbours, which
  is why exact-ID recall UNDERSTATES quantization quality for RAG.
  Measure both on YOUR corpus before shipping — the trade is corpus-dependent.


### 📝 Nhận xét — Chất lượng sau lượng tử hoá

- **recall@10 = 0,904** (≥ 0,80 ✅): int8 giữ 90% **đúng ID** trong top-10 so với float32 (ground truth), trên 100 truy vấn.
- **Topic fidelity = 1,000** (≥ 0,95 ✅): **100%** kết quả int8 vẫn đúng topic. Các ID "mất" là **hoán đổi giữa những hàng xóm gần tương đương** ⇒ recall theo ID chính xác *đánh giá thấp* chất lượng int8 cho RAG, nơi điều quan trọng là tài liệu lấy về có đúng chủ đề không.
- Lưu ý: truy vấn nằm trong corpus nên luôn khớp chính nó trong cả hai top-10; recall trên 9 hàng xóm thật ≈ (9,04 − 1)/9 ≈ **0,89** — vẫn trên ngưỡng.
- Đánh đổi: tiết kiệm **83%** dung lượng đổi lấy ~10% sai khác ID. Corpus tổng hợp có topic tách bạch nên fidelity cao — **phải đo lại trên corpus thật** trước khi dùng.


## 5. The lifecycle bug — the real reason embeddings belong in the table

> *"When a row is deleted, expires, is edited or reprocessed, the embedding
> must follow exactly that lifecycle. Every warehouse → vector-DB sync
> pipeline is a lifecycle-skew bug waiting to happen — and the moment you get
> a right-to-forget request, it becomes a **compliance** bug."*

Let's produce that bug on purpose.

First, a nightly sync builds an external vector index (a copy):

In [12]:
EXTERNAL = path("scratch", "vector_index_external")
reset(EXTERNAL)
write_deltalake(EXTERNAL, pa.table({
    "doc_id": docs.column("doc_id"),
    "emb": docs.column("emb"),
}), mode="overwrite")
print(f"External index synced: {DeltaTable(EXTERNAL).count():,} vectors")

External index synced: 2,000 vectors


### 📝 Nhận xét — Index ngoài là một bản sao

- Một job đồng bộ hằng đêm chép **2.000 vector** sang index ngoài (mô phỏng vector DB). Từ thời điểm này có **hai bản sự thật** — và chúng sẽ lệch nhau ngay khi bảng gốc thay đổi.


### A data subject exercises their right to erasure

`user_042` asks for deletion. We delete from the lakehouse — the
system-of-record — as we should.

In [13]:
SUBJECT = "user_042"
INTABLE = path("scratch", "docs_intable")
reset(INTABLE)
write_deltalake(INTABLE, docs, mode="overwrite")

dt = DeltaTable(INTABLE)
victim_ids = con.sql(f"SELECT doc_id FROM docs WHERE subject_id = '{SUBJECT}'").fetchall()
victim_ids = [r[0] for r in victim_ids]
dt.delete(f"subject_id = '{SUBJECT}'")

print(f"Erasure request for {SUBJECT}: {len(victim_ids)} docs")
print(f"lakehouse rows: {docs.num_rows:,} → {DeltaTable(INTABLE).count():,}")
print(f"external index rows: {DeltaTable(EXTERNAL).count():,}   ← untouched")

Erasure request for user_042: 8 docs


lakehouse rows: 2,000 → 1,992
external index rows: 2,000   ← untouched


### 📝 Nhận xét — Yêu cầu xoá dữ liệu

- `user_042` yêu cầu xoá: **8 tài liệu**. Ta xoá đúng chỗ — ở lakehouse (system-of-record): **2.000 → 1.992** dòng.
- Index ngoài vẫn **2.000** — không có gì báo cho nó biết.


### Now query both

In [14]:
victim_vec = emb[victim_ids[0]].tolist()

con.register("intable", DeltaTable(INTABLE).to_pyarrow_table())
con.register("external", DeltaTable(EXTERNAL).to_pyarrow_table())

in_hits = con.sql(f"""SELECT count(*) FROM intable
                      WHERE doc_id IN ({','.join(map(str, victim_ids))})""").fetchone()[0]
ex_hits = con.sql(f"""SELECT count(*) FROM external
                      WHERE doc_id IN ({','.join(map(str, victim_ids))})""").fetchone()[0]

print(f"Erased docs still retrievable from the lakehouse:      {in_hits}")
print(f"Erased docs still retrievable from the external index: {ex_hits}   ← VIOLATION")
print(f"""
The external index will happily return {SUBJECT}'s content to a RAG prompt
until the next sync — and if the sync is one-way upsert (the common case),
*forever*, because deletes are the operation sync pipelines forget.
""")

Erased docs still retrievable from the lakehouse:      0
Erased docs still retrievable from the external index: 8   ← VIOLATION

The external index will happily return user_042's content to a RAG prompt
until the next sync — and if the sync is one-way upsert (the common case),
*forever*, because deletes are the operation sync pipelines forget.



In [15]:
# DELETE ở version hiện tại có xoá thật không? Thử time travel về version trước khi xoá
dt_in = DeltaTable(INTABLE)
for h in dt_in.history():
    m = h.get("operationMetrics", {})
    print(f"  v{h['version']} {h['operation']:<7} deleted_rows={m.get('num_deleted_rows', '-')}  "
          f"copied_rows={m.get('num_copied_rows', '-')}  files +{m.get('num_added_files')} -{m.get('num_removed_files')}")
con.register("intable_v0", DeltaTable(INTABLE, version=0).to_pyarrow_table())
v0_hits = con.sql(f"SELECT count(*) FROM intable_v0 WHERE subject_id = '{SUBJECT}'").fetchone()[0]
print(f"\n{SUBJECT} rows at CURRENT version : {in_hits}")
print(f"{SUBJECT} rows via time travel v0: {v0_hits}   ← vẫn đọc được")
print(f"parquet files on disk: {len(list(Path(INTABLE).glob('*.parquet')))}  |  referenced now: {len(dt_in.file_uris())}")

  v1 DELETE  deleted_rows=8  copied_rows=1992  files +1 -1
  v0 WRITE   deleted_rows=-  copied_rows=-  files +1 -0

user_042 rows at CURRENT version : 0
user_042 rows via time travel v0: 8   ← vẫn đọc được
parquet files on disk: 2  |  referenced now: 1


### 📝 Nhận xét — Lỗi vòng đời được tái hiện, và một lớp sâu hơn

- **0 hit trong bảng, 8 hit trong index ngoài** ⇒ lỗi vòng đời tái hiện ✅. Một prompt RAG truy vấn index ngoài vẫn nhận nội dung của người đã yêu cầu xoá ⇒ đây là **lỗi tuân thủ**, không chỉ lỗi kỹ thuật.
- Nếu job đồng bộ là **upsert một chiều** (phổ biến nhất), các bản ghi đã xoá ở nguồn **không bao giờ** bị xoá ở đích — upsert chỉ mang insert/update, không mang delete.
- **Lớp sâu hơn (cell trên):** `DELETE` của Delta là **copy-on-write** — ghi lại 1.992 dòng sang file mới (`copied_rows=1992`) và tombstone file cũ. File cũ **vẫn trên đĩa** (2 file, chỉ 1 được tham chiếu) ⇒ **time travel về v0 vẫn đọc được đủ 8 tài liệu của `user_042`**.
- Nghĩa là: **xoá ở version hiện tại ≠ xoá thật**. Để đáp ứng right-to-erasure phải **DELETE → VACUUM** sau retention (NB6), và mọi bản sao phái sinh (index, cache, bảng Gold) cũng phải nhận được sự kiện xoá.


### The correct propagation mechanism: Change Data Feed

If you *must* keep a derived index, do not re-sync the whole table. Read the
**change feed** so deletes propagate as first-class events.

In [16]:
CDF_TABLE = path("scratch", "docs_cdf")
reset(CDF_TABLE)
write_deltalake(CDF_TABLE, pa.table({"doc_id": docs.column("doc_id"),
                                     "subject_id": docs.column("subject_id")}),
                mode="overwrite", configuration={"delta.enableChangeDataFeed": "true"})
DeltaTable(CDF_TABLE).delete(f"subject_id = '{SUBJECT}'")

cdf = DeltaTable(CDF_TABLE).load_cdf(starting_version=1).read_all()
changes = cdf.column("_change_type").to_pylist()
deletes = [t for t in changes if t == "delete"]
print(f"CDF rows since v1: {len(changes)}   deletes: {len(deletes)}")
print(f"Delete events carry the doc_ids to evict: {cdf.column('doc_id').to_pylist()[:5]} ...")
print("""
That is the contract: the index subscribes to deletes instead of guessing.
Best of all is not needing the sync — keep the vector in the row (§2 above)
and the lifecycle is enforced by the table itself.
""")

CDF rows since v1: 8   deletes: 8


Delete events carry the doc_ids to evict: [42, 292, 542, 792, 1042] ...

That is the contract: the index subscribes to deletes instead of guessing.
Best of all is not needing the sync — keep the vector in the row (§2 above)
and the lifecycle is enforced by the table itself.



### 📝 Nhận xét — Cơ chế lan truyền đúng: Change Data Feed

- Bật `delta.enableChangeDataFeed` ⇒ mỗi thay đổi được ghi thành sự kiện có `_change_type`. Sau khi xoá `user_042`, CDF từ v1 có **8 dòng, cả 8 là `delete`** (khớp 8 tài liệu), mang `doc_id` cần gỡ khỏi index (42, 292, 542, 792, 1042, …).
- Hợp đồng đúng: index **đăng ký nhận sự kiện xoá** thay vì đồng bộ lại toàn bộ và đoán. Tốt nhất là **không cần đồng bộ**: giữ vector trong chính dòng dữ liệu (§2–§3) ⇒ vòng đời của embedding do bảng thực thi — xoá dòng là xoá vector.


## ✅ NB7 pass criteria

| Check | Target |
|---|---|
| Random-access amplification measured | ≥ 5× more bytes for inline blob |
| int8 quantization | ≥ 3× smaller on disk |
| int8 recall@10 | ≥ 0.80 vs float32 (typically ~0.90) |
| int8 topic fidelity | ≥ 0.95 — the metric that matters for RAG |
| Semantic search returns same-topic neighbours | top-5 majority share the query's topic |
| Lifecycle bug reproduced | 0 hits in-table, > 0 hits in the external index |
| CDF emits delete events | ≥ 1 `delete` in the change feed |

In [17]:
top_topics = [h[2] for h in hits]
query_topic = docs.column("topic")[7].as_py()

checks = {
    "random-access amplification ≥ 5x": AMPLIFICATION >= 5,
    "int8 ≥ 3x smaller":                du(F32) / max(du(I8), 1) >= 3,
    "int8 recall@10 ≥ 0.80":            recall >= 0.80,
    "int8 topic fidelity ≥ 0.95":       topic_fidelity >= 0.95,
    "top-5 share query topic":          top_topics.count(query_topic) >= 3,
    "lifecycle bug reproduced":         in_hits == 0 and ex_hits > 0,
    "CDF emits delete events":          len(deletes) == len(victim_ids),
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB7 incomplete — see FAIL rows above"
print("\nNB7 complete.")

  [PASS] random-access amplification ≥ 5x
  [PASS] int8 ≥ 3x smaller
  [PASS] int8 recall@10 ≥ 0.80
  [PASS] int8 topic fidelity ≥ 0.95
  [PASS] top-5 share query topic
  [PASS] lifecycle bug reproduced
  [PASS] CDF emits delete events

NB7 complete.


### 🎯 Kết luận NB7

Multimodal phá vỡ giả định "dòng nhỏ, đọc tuần tự": blob inline **không** làm hại scan phân tích (column pruning: 1,2 KB / 12,5 MB) nhưng làm truy cập ngẫu nhiên **khuếch đại 200×** vì đơn vị I/O là row group. Embedding là một cột như mọi cột: int8 nhỏ hơn **5,8×** (nhờ dictionary encoding) với recall@10 **0,904** và topic fidelity **1,000**; semantic search là một câu SQL và lọc được theo license/consent trong cùng truy vấn (doc 7 bị loại). Lý do quan trọng nhất để giữ vector trong bảng là **vòng đời**: index ngoài vẫn trả 8 tài liệu đã bị xoá — chỉ CDF (hoặc không sync) mới mang được sự kiện delete. Và ngay cả trong lakehouse, xoá ở version hiện tại chưa phải xoá thật cho tới khi VACUUM.
